## OBJETIVO DEL CUADERNO
En este código vamos a importar dos bases de datos, una correspondiente a la clasificación regional M49 de ONU, y otra que permite detectar cuales
son los países miembros de ONU. Vamos a descartar los países no miembros, para quedarnos con países soberanos. De esta manera evitamos sumar unidades
económicas que tienen muy baja cobertura de datos en las fuentes temporales. Además la clasificación regional propuesta por oNU (M49) presenta una
desagregación que consideramos óptima para captar fenómenos regionales.

## Importamos bases de ONU: países miembros y clasificación m49

In [1]:
import pandas as pd

df_miembros_onu = pd.read_excel("193miembros_onu.xlsx")
df_m49_onu = pd.read_excel("M49_onu.xlsx")

df_miembros_onu.head()

,Member State,M49 Code,ISO Code,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI


In [2]:
df_miembros_onu.columns

Index(['Member State', 'M49 Code', 'ISO Code', 'Other Names',
       'Earlier or Later Name', 'Earlier (a) or Later (b)', 'Geographic Term'],
      dtype='str')

In [3]:
df_m49_onu.columns

Index(['Region Code', 'Region Name', 'Sub-region Code', 'Sub-region Name',
       'Intermediate Region Code', 'Intermediate Region Name',
       'Country or Area', 'M49 Code', 'ISO-alpha2 Code', 'ISO-alpha3 Code',
       'Least Developed Countries (LDC)',
       'Land Locked Developing Countries (LLDC)',
       'Small Island Developing States (SIDS)'],
      dtype='str')

## Renombrar columnas

In [4]:
# Bloque 1: Renombrar columna clave en ambos df para unificar criterio de merge
df_m49_onu = df_m49_onu.rename(columns={'ISO-alpha3 Code': 'ISO-alpha3'})
df_miembros_onu = df_miembros_onu.rename(columns={'ISO Code': 'ISO-alpha3'})

## Unificamos df


In [5]:
# Bloque 2: Seleccionar únicamente las columnas de df_m49_onu que se van a incorporar
cols_m49 = [
    'ISO-alpha3',
    'Country or Area',
    'Region Code',
    'Region Name',
    'Sub-region Code',
    'Sub-region Name',
    'ISO-alpha2 Code'
]
df_m49_sub = df_m49_onu[cols_m49].copy()

In [6]:
# Bloque 3: Merge — df_miembros_onu como referencia (left join)
df_regiones_miembros_onu = df_miembros_onu.merge(
    df_m49_sub,
    on='ISO-alpha3',
    how='left',
    validate='one_to_one'  # asegura que no haya duplicados en ninguno de los dos df
)

print(df_regiones_miembros_onu.shape)
df_regiones_miembros_onu.head()

(193, 13)


,Member State,M49 Code,ISO-alpha3,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term,Country or Area,Region Code,Region Name,Sub-region Code,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES,United States of America,19.0,Americas,21.0,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA,Australia,9.0,Oceania,53.0,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI,Djibouti,2.0,Africa,202.0,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA,Ghana,2.0,Africa,202.0,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI,Kiribati,9.0,Oceania,57.0,Micronesia,KI


In [7]:
# Bloque 4: Formatear Region Code y Sub-region Code como categóricas, sin alterar los valores
df_regiones_miembros_onu['Region Code'] = df_regiones_miembros_onu['Region Code'].astype('Int64').astype('category')
df_regiones_miembros_onu['Sub-region Code'] = df_regiones_miembros_onu['Sub-region Code'].astype('Int64').astype('category')

df_regiones_miembros_onu.dtypes

Member State                     str
M49 Code                       int64
ISO-alpha3                       str
Other Names                      str
Earlier or Later Name            str
Earlier (a) or Later (b)         str
Geographic Term                  str
Country or Area                  str
Region Code                 category
Region Name                      str
Sub-region Code             category
Sub-region Name                  str
ISO-alpha2 Code                  str
dtype: object

In [8]:
df_regiones_miembros_onu.shape

(193, 13)

In [9]:
df_regiones_miembros_onu.head()

,Member State,M49 Code,ISO-alpha3,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term,Country or Area,Region Code,Region Name,Sub-region Code,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES,United States of America,19,Americas,21,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA,Australia,9,Oceania,53,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA,Ghana,2,Africa,202,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI,Kiribati,9,Oceania,57,Micronesia,KI


## Organizar el nuevo df unificado

In [10]:
df_regiones_miembros_onu.columns

Index(['Member State', 'M49 Code', 'ISO-alpha3', 'Other Names',
       'Earlier or Later Name', 'Earlier (a) or Later (b)', 'Geographic Term',
       'Country or Area', 'Region Code', 'Region Name', 'Sub-region Code',
       'Sub-region Name', 'ISO-alpha2 Code'],
      dtype='str')

In [11]:
#Renombrar columnas 
df_regiones_miembros_onu = df_regiones_miembros_onu.rename(columns={
    'M49 Code': 'M49_country',
    'Region Code': 'M49_region',
    'Sub-region Code': 'M49_subregion'
})

In [12]:
#eliminar columnas poco útiles
df_regiones_miembros_onu = df_regiones_miembros_onu.drop(
    columns=['Earlier (a) or Later (b)', 'Earlier or Later Name', 'Geographic Term']
)

In [13]:
#Reubicar 'ISO-alpha2 Code' inmediatamente después de 'ISO-alpha3'
cols = df_regiones_miembros_onu.columns.tolist()
cols.remove('ISO-alpha2 Code')
pos = cols.index('ISO-alpha3') + 1
cols.insert(pos, 'ISO-alpha2 Code')


In [14]:
df_regiones_miembros_onu.head()

,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI


## Vamos a sumar la base de COW CODE

De aquí https://correlatesofwar.org/wp-content/uploads/COW-country-codes.csv fueron extraídas las nomenclaturas de cow code para incorporarlas en la base ya estandarizada de mimebros y regiones ONU

In [15]:
# Importación del estándar Correlates of War (COW)
df_cow = pd.read_csv("COW-country-codes.csv", lineterminator="\r")

print("Filas originales:", len(df_cow))

df_cow = df_cow.drop_duplicates().reset_index(drop=True)

print("Filas tras eliminar duplicados exactos:", len(df_cow))
print("StateAbb únicos:", df_cow["StateAbb"].nunique())
print("CCode únicos:", df_cow["CCode"].nunique())

df_cow.head()

Filas originales: 243
Filas tras eliminar duplicados exactos: 217
StateAbb únicos: 217
CCode únicos: 217


,StateAbb,CCode,StateNme
0,USA,2,United States of America
1,CAN,20,Canada
2,BHM,31,Bahamas
3,CUB,40,Cuba
4,HAI,41,Haiti


### Matcheamos codificacion CowCode de VDEM con ONU

Vamos a normalizar nombres de paises y ver si podemos matchear con los que tenemos en la base de ONU

In [16]:
import re
import unicodedata

def normalizar_nombre(texto):
    """Normaliza nombres de país: minúsculas, sin tildes, sin paréntesis, sin puntuación."""
    if pd.isna(texto):
        return ''
    texto = str(texto).lower().strip()
    texto = re.sub(r'\([^)]*\)', '', texto)
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    texto = re.sub(r'[^a-z0-9 ]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Normalización de nombres en ambas bases
df_cow['nombre_norm'] = df_cow['StateNme'].apply(normalizar_nombre)
df_regiones_miembros_onu['nombre_norm_principal'] = df_regiones_miembros_onu['Member State'].apply(normalizar_nombre)
df_regiones_miembros_onu['nombre_norm_alt'] = df_regiones_miembros_onu['Other Names'].apply(normalizar_nombre)

# Diccionario de búsqueda: nombre normalizado -> ISO-alpha3
onu_lookup = {}
for _, fila in df_regiones_miembros_onu.iterrows():
    for col in ['nombre_norm_principal', 'nombre_norm_alt']:
        if fila[col]:
            onu_lookup[fila[col]] = fila['ISO-alpha3']

# Asignación de ISO-alpha3 por coincidencia de nombre
df_cow['ISO_match'] = df_cow['nombre_norm'].map(onu_lookup)

print(f"Matcheados automáticamente: {df_cow['ISO_match'].notna().sum()} / {len(df_cow)}")

# Casos sin match, para revisión manual explícita
no_match = df_cow[df_cow['ISO_match'].isna()][['StateAbb', 'CCode', 'StateNme']].reset_index(drop=True)
no_match

Matcheados automáticamente: 175 / 217


,StateAbb,CCode,StateNme
0,USA,2,United States of America
1,SVG,57,St. Vincent and the Grenadines
2,AAB,58,Antigua & Barbuda
3,SKN,60,St. Kitts and Nevis
4,HAN,240,Hanover
5,BAV,245,Bavaria
6,GFR,260,German Federal Republic
7,GDR,265,German Democratic Republic
8,BAD,267,Baden
9,SAX,269,Saxony


In [17]:
#Ahora vamos a ver el match en sentido inverso: de los que estan en el df con miembros de onu, cuales estan matcheados
cow_nombres = set(df_cow['nombre_norm'])

def busca_match_en_cow(fila):
    if fila['nombre_norm_principal'] in cow_nombres:
        return True
    if fila['nombre_norm_alt'] and fila['nombre_norm_alt'] in cow_nombres:
        return True
    return False

df_regiones_miembros_onu['match_cow'] = df_regiones_miembros_onu.apply(busca_match_en_cow, axis=1)

print(f"Matcheados: {df_regiones_miembros_onu['match_cow'].sum()} / {len(df_regiones_miembros_onu)}")

no_match_onu = df_regiones_miembros_onu[~df_regiones_miembros_onu['match_cow']][
    ['Member State', 'Other Names', 'ISO-alpha3']
].reset_index(drop=True)
no_match_onu

Matcheados: 175 / 193


,Member State,Other Names,ISO-alpha3
0,United States,"USA, U.S.A., United States of America",USA
1,Lao People's Democratic Republic,"People's Democratic Republic of Laos, R��publi...",LAO
2,Syrian Arab Republic,NaN,SYR
3,Saint Kitts and Nevis,St. Kitts-Nevis,KNA
4,Russian Federation,"Russia, Russie, Rusia, Russia (Federation), Ro...",RUS
5,Cote d'Ivoire,"Territoire de la Cote d'Ivoire, Republique de ...",CIV
6,Saint Vincent and the Grenadines,NaN,VCT
7,Democratic People's Republic of Korea,"North Korea, Korean People's Republic, DPRK",PRK
8,Viet Nam,"Socialist Republic of Viet Nam, Cong Hoa Xa H�...",VNM
9,Antigua and Barbuda,NaN,ATG


In [18]:
# Diccionario inverso: nombre normalizado -> CCode de COW
cow_code_lookup = dict(zip(df_cow['nombre_norm'], df_cow['CCode']))

def obtener_cow_code(fila):
    if fila['nombre_norm_principal'] in cow_code_lookup:
        return cow_code_lookup[fila['nombre_norm_principal']]
    if fila['nombre_norm_alt'] and fila['nombre_norm_alt'] in cow_code_lookup:
        return cow_code_lookup[fila['nombre_norm_alt']]
    return pd.NA

df_regiones_miembros_onu = df_regiones_miembros_onu.copy()
df_regiones_miembros_onu['cow_code'] = df_regiones_miembros_onu.apply(obtener_cow_code, axis=1)

print(f"Con cow_code: {df_regiones_miembros_onu['cow_code'].notna().sum()} / {len(df_regiones_miembros_onu)}")
df_regiones_miembros_onu.head(10)

Con cow_code: 175 / 193


,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code,nombre_norm_principal,nombre_norm_alt,match_cow,cow_code
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US,united states,usa u s a united states of america,False,<NA>
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU,australia,commonwealth of australia,True,900
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ,djibouti,republic of djibouti,True,522
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH,ghana,republic of ghana,True,452
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI,kiribati,republic of kiribati,True,946
5,Iran (Islamic Republic of),364,IRN,Islamic Republic of Iran,Iran (Islamic Republic of),142,Asia,34,Southern Asia,IR,iran,islamic republic of iran,True,630
6,Japan,392,JPN,NaN,Japan,142,Asia,30,Eastern Asia,JP,japan,,True,740
7,Kuwait,414,KWT,State of Kuwait,Kuwait,142,Asia,145,Western Asia,KW,kuwait,state of kuwait,True,690
8,Rwanda,646,RWA,"Republic of Rwanda, Rwandese Republic",Rwanda,2,Africa,202,Sub-Saharan Africa,RW,rwanda,republic of rwanda rwandese republic,True,517
9,Saint Lucia,662,LCA,St. Lucia,Saint Lucia,19,Americas,419,Latin America and the Caribbean,LC,saint lucia,st lucia,True,56


In [19]:
#LISTA de países sin cow_code
paises_sin_cow_code = df_regiones_miembros_onu.loc[
    df_regiones_miembros_onu['cow_code'].isna(), 'Member State'
].tolist()

paises_sin_cow_code

['United States',
 "Lao People's Democratic Republic",
 'Syrian Arab Republic',
 'Saint Kitts and Nevis',
 'Russian Federation',
 "Cote d'Ivoire",
 'Saint Vincent and the Grenadines',
 "Democratic People's Republic of Korea",
 'Viet Nam',
 'Antigua and Barbuda',
 'Republic of Moldova',
 'Timor-Leste',
 'Serbia',
 'Cabo Verde',
 'Czechia',
 'North Macedonia',
 'T��rkiye',
 'Eswatini']

### los que faltan, los asignamos manualmente

Extrajimos manualmente estos valores faltantes de la base de vdem descargada (hecho en cuaderno 4). Y los completamos a continuación.

TENER EN CUENTA: Serbia (345): es el mismo código que Yugoslavia.

In [20]:
# Asignación manual verificada contra COW-country-codes.csv 
cow_code_manual = {
    'United States': 2,
    "Lao People's Democratic Republic": 812,
    'Syrian Arab Republic': 652,
    'Saint Kitts and Nevis': 60,
    'Russian Federation': 365,
    "Cote d'Ivoire": 437,
    'Saint Vincent and the Grenadines': 57,
    "Democratic People's Republic of Korea": 731,
    'Viet Nam': 816,
    'Antigua and Barbuda': 58,
    'Republic of Moldova': 359,
    'Timor-Leste': 860,
    'Serbia': 345,          # mismo CCode histórico que Yugoslavia en COW; confirmado vía V-Dem
    'Cabo Verde': 402,
    'Czechia': 316,
    'North Macedonia': 343,
    'Türkiye': 640,
    'Eswatini': 572,
}

mask_faltantes = df_regiones_miembros_onu['cow_code'].isna()

df_regiones_miembros_onu.loc[mask_faltantes, 'cow_code'] = (
    df_regiones_miembros_onu.loc[mask_faltantes, 'Member State'].map(cow_code_manual)
)

print(f"Faltantes tras asignación manual: {df_regiones_miembros_onu['cow_code'].isna().sum()}")
df_regiones_miembros_onu[df_regiones_miembros_onu['Member State'].isin(cow_code_manual.keys())][['Member State', 'cow_code']]

Faltantes tras asignación manual: 1


,Member State,cow_code
0,United States,2.0
25,Lao People's Democratic Republic,812.0
34,Syrian Arab Republic,652.0
35,Saint Kitts and Nevis,60.0
37,Russian Federation,365.0
57,Cote d'Ivoire,437.0
67,Saint Vincent and the Grenadines,57.0
92,Democratic People's Republic of Korea,731.0
121,Viet Nam,816.0
123,Antigua and Barbuda,58.0


In [21]:
# Diagnóstico: ver cuál quedó sin asignar
pendiente = df_regiones_miembros_onu[df_regiones_miembros_onu['cow_code'].isna()]
print(pendiente[['Member State', 'ISO-alpha3']])

    Member State ISO-alpha3
144     T��rkiye        TUR


In [22]:
df_regiones_miembros_onu.loc[
    df_regiones_miembros_onu['ISO-alpha3'] == 'TUR', 'cow_code'
] = 640

print(f"Faltantes tras corrección: {df_regiones_miembros_onu['cow_code'].isna().sum()}")

Faltantes tras corrección: 0


## Exporto el df en excel

In [23]:
#Primero quiero chequear que el df quedó como quiero
df_regiones_miembros_onu.head()

,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code,nombre_norm_principal,nombre_norm_alt,match_cow,cow_code
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US,united states,usa u s a united states of america,False,2.0
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU,australia,commonwealth of australia,True,900
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ,djibouti,republic of djibouti,True,522
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH,ghana,republic of ghana,True,452
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI,kiribati,republic of kiribati,True,946


In [24]:
#Voy a renombrar y eliminar algunas columnas
df_regiones_miembros_onu = df_regiones_miembros_onu.rename(
    columns={'cow_code': 'cow_code_countryVdem'}
)

df_regiones_miembros_onu = df_regiones_miembros_onu.drop(columns=['match_cow'])

print(df_regiones_miembros_onu.columns.tolist())

['Member State', 'M49_country', 'ISO-alpha3', 'Other Names', 'Country or Area', 'M49_region', 'Region Name', 'M49_subregion', 'Sub-region Name', 'ISO-alpha2 Code', 'nombre_norm_principal', 'nombre_norm_alt', 'cow_code_countryVdem']


In [25]:
# Exportar df_regiones_miembros_onu a Excel  (DESCOMENTAR PARA DESCARGAR)
#df_regiones_miembros_onu.to_excel("df_regiones_miembros_onu.xlsx", index=False)

## Comparando con el universo real de países de VDEM

Los Cow Code con los que trabajamos hasta acá fueron descargados de una fuente oficial para incorporarlos a la estandarización ONU.
Descargamos los Cow Code porque ya habíamos visto que en VDEM se trabajaba con esa codificación, pero ahora vamos a ver efectivamente
si podemos obtener match

In [26]:
df_paises_vdem = pd.read_excel("df_paises_vdem.xlsx")

codigos_vdem_reales = set(
    df_paises_vdem["COWcode"].dropna().astype(int)
)

print(f"Países únicos en df_paises_vdem: {df_paises_vdem.shape[0]}")
print(f"Códigos COW válidos (no nulos) en V-Dem: {len(codigos_vdem_reales)}")

Países únicos en df_paises_vdem: 211
Códigos COW válidos (no nulos) en V-Dem: 196


Los que no tienen código COW code en la base real de VDEM son: 

Brunswick

Hamburg

Hesse-Darmstadt

Hong Kong

Nassau

Oldenburg

Palestine/British Mandate

Palestine/Gaza

Palestine/West Bank

Papal States

Piedmont-Sardinia

Saxe-Weimar-Eisenach

Somaliland

Tuscany

Zanzibar


Ninguno de ellos son mimebros de ONU